## Example for extracting data for GPT prompting

### This is not the final/complete code but more about how to get the desired data from the table

In [1]:
import sqlite3
import pandas as pd
from tqdm import tqdm
import numpy as np
import copy
import tiktoken
import openai 
from gpt_cost_estimator import CostEstimator
import os
from openai import AzureOpenAI
import configparser
import json
import time
import pydantic
from pydantic import Field
from typing import Literal, List


from enum import Enum
from pydantic import BaseModel
import outlines
#from outlines.models.openai import OpenAI #OpenAIChatCompletion
#from outlines.generate import pydantic
from outlines.models.openai import OpenAI, OpenAIConfig
#from outlines.integrations.pydantic import from_pydantic

from pydantic import ValidationError

/home/kaire/anaconda3/envs/gpt4/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [45]:
pd.set_option('display.max_colwidth', None)

# OSA I : teha kui pole olemas csv faili andmetega

## andmetabelid

In [42]:
filename = "../drive_data/v33_koondkorpus_transaktsioonid.db"
conn = sqlite3.connect(filename)
cursor = conn.cursor()

## graafiku punktide info

In [43]:
query = """SELECT verb, verb_compound, morph_case, log2_ratio, level,unique_lemmas, ann_unique_lemmas, 
            not_ann_unique_lemmas, olulisus, my_tag, other_tags, annotated, not_annotated, verb_case_count
            FROM lines_class_info4
            """

class_info = pd.read_sql(query, conn)
class_info

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
0,aasima,,ad,-9.965784,-,1,NaN,5.0,-,0,1,1,7,8
1,abistama,,abl,-9.965784,-,1,NaN,4.0,-,0,1,1,6,7
2,abistama,,all,-9.965784,-,1,NaN,12.0,-,0,1,1,15,16
3,adresseerima,,in,-9.965784,-,1,NaN,3.0,-,0,1,1,6,7
4,aeglustama,,all,-9.965784,-,1,NaN,7.0,-,0,1,1,8,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21264,õnnestuma,,in,1.227736,-,312,209.0,529.0,-,541,231,772,1286,2058
21265,õppima,,in,4.905379,n90,547,465.0,968.0,0.0,5724,191,5915,5848,11763
21266,ütlema,,ad,-5.088166,n10,459,112.0,1118.0,0.0,362,12314,12676,9790,22466
21267,ütlema,,el,0.974385,n70,629,337.0,1192.0,0.97997,949,483,1432,2839,4271


## näitelausete tabel

In [17]:
#query = f"SELECT * FROM spatial_obl"

#spatial_obl = pd.read_sql(query, conn)
#spatial_obl

## võtta ainult n80 tsooni punktid

In [44]:
filtered_class = class_info[class_info["level"]=="n80"]
filtered_class = filtered_class.sort_values(["olulisus"])

In [45]:
filtered_class['olulisus'] = filtered_class['olulisus'].astype(float)

In [46]:
filtered_class

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
21249,valitsema,,in,3.044781,n80,677,551.0,1234.0,0.00000,1865,226,2091,3776,5867
21223,toimuma,,in,2.873490,n80,2200,1871.0,4539.0,0.00000,22095,3015,25110,24937,50047
19354,treenima,,in,3.900242,n80,142,124.0,198.0,0.00000,433,29,462,387,849
19486,õpetama,,in,3.626783,n80,219,181.0,317.0,0.00000,630,51,681,788,1469
19531,kasvama,üles,in,3.798366,n80,172,158.0,136.0,0.00000,320,23,343,391,734
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18864,põgenema,,adit,4.285402,n80,83,72.0,90.0,0.00008,351,18,369,231,600
19931,elutsema,,in,4.554589,n80,74,70.0,105.0,0.00008,94,4,98,150,248
19940,müüma,,ill,4.610498,n80,74,67.0,53.0,0.00008,171,7,178,84,262
19943,pöörduma,,adit,4.643856,n80,74,63.0,152.0,0.00008,325,13,338,416,754


In [47]:
only_zero = filtered_class[filtered_class["olulisus"]==0] # 29 juhtu
only_zero

,verb,verb_compound,morph_case,log2_ratio,level,unique_lemmas,ann_unique_lemmas,not_ann_unique_lemmas,olulisus,my_tag,other_tags,annotated,not_annotated,verb_case_count
21249,valitsema,,in,3.044781,n80,677,551.0,1234.0,0.0,1865,226,2091,3776,5867
21223,toimuma,,in,2.873490,n80,2200,1871.0,4539.0,0.0,22095,3015,25110,24937,50047
19354,treenima,,in,3.900242,n80,142,124.0,198.0,0.0,433,29,462,387,849
19486,õpetama,,in,3.626783,n80,219,181.0,317.0,0.0,630,51,681,788,1469
19531,kasvama,üles,in,3.798366,n80,172,158.0,136.0,0.0,320,23,343,391,734
19708,saabuma,,el,3.516847,n80,660,513.0,596.0,0.0,2358,206,2564,1240,3804
19841,avama,,in,2.904422,n80,641,569.0,790.0,0.0,3212,429,3641,3231,6872
19918,varjama,,in,3.778973,n80,161,144.0,197.0,0.0,302,22,324,491,815
20041,varastama,,el,3.754888,n80,159,139.0,244.0,0.0,324,24,348,626,974
20164,korraldama,,in,2.725225,n80,685,620.0,743.0,0.0,3141,475,3616,3182,6798



example_row = only_zero.iloc[4]

v = example_row["verb"]
v_c = example_row["verb_compound"]
m_c = example_row["morph_case"]

# transaction_head.form as head_form, lemma, spatial_obl.form as verb_form, verb, verb_compound, morph_case, sentence_id, sentence, phrase
query = f"""SELECT head_id, head_form, head_lemma, tbl2.form as verb_form, tbl1.verb, tbl1.verb_compound, 
            tbl1.morph_case, tbl1.sentence_id, tbl1.sentence, tbl2.phrase, tbl1.ekilex_tag

            FROM (
            SELECT head_id, form as head_form, lemma as head_lemma, verb, verb_compound, morph_case, 
            sentence_id, sentence, ekilex_tag 
            FROM spatial_obl 
            where 
            verb = '{v}' and 
            verb_compound='{v_c}' and 
            morph_case='{m_c}'
            and ekilex_tag = 'location'
            ) as tbl1

            join

            (SELECT * from transaction_head) as tbl2 on 
            tbl1.verb = tbl2.verb and 
            tbl1.verb_compound = tbl2.verb_compound and 
            tbl1.sentence_id = tbl2.sentence_id
            """

spatial_obl_ex1 = pd.read_sql(query, conn)

In [51]:
#spatial_obl_ex1

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag
0,54396,Aafrikas,Aafrika,kasvanud,kasvama,üles,in,31692,Aga Aafrikas üles kasvanud poisina ei teadnud ...,Aafrikas üles kasvanud,location
1,105869,põgenikelaagrites,põgenikelaager,kasvavad,kasvama,üles,in,61570,Kuni aga tõsimeeli taotleb kompromisslahendust...,mille jooksul kasvavad põgenikelaagrites üles ...,location
2,115977,USA-s,USA,kasvas,kasvama,üles,in,67436,"Meiegi ei saa olla ükskõiksed , eriti mitte Vä...",USA-s kasvas üles osa,location
3,137921,Tallinnas,Tallinn,kasvas,kasvama,üles,in,81015,I 1886 ) ja kasvas üles Tallinnas kasuisa Osca...,kasvas üles Tallinnas perekonnas,location
4,181685,Avinurmes,Avinurme,kasvanud,kasvama,üles,in,107978,"Olen Avinurmes üles kasvanud ja tean , et seal...",Olen Avinurmes üles kasvanud,location
...,...,...,...,...,...,...,...,...,...,...,...
315,27929765,Tartus,Tartu,kasvanud,kasvama,üles,in,18429554,Teistkordselt toob Liivimaalt pärit ja Tartus ...,Tartus üles kasvanud,location
316,28521921,Vändras,Vändra,kasvanud,kasvama,üles,in,18885996,"Marek Helm on Saaremaal sündinud , Vändras üle...",Vändras üles kasvanud,location
317,28701500,Eestis,Eesti,kasvanud,kasvama,üles,in,19018175,"Võin kinnitada , et vaatamata sellele , et väg...",ajal Eestis üles kasvanud,location
318,28742481,Eestis,Eesti,kasvanud,kasvama,üles,in,19046659,Aga kuidasmoodi te selgitate sellist olukorda ...,kes ei ole üles kasvanud Eestis,location


## võtta spatial_obl tabelist näitelaused koos vajaliku infoga

kui tahta ainult location näiteid, siis peaks from sees olema lisatingimus

verb = '{v}' and 
verb_compound='{v_c}' and 
morph_case='{m_c}'
and ekilex_tag = 'location'

In [62]:
# nt class_info tabelist esimene ja sellele vastavad näited

#df1.append(df2, ignore_index=True)

big_df = pd.DataFrame()

for i in range(len(only_zero)):
    example_row = only_zero.iloc[i]

    #example_row = filtered_class.iloc[4]
    #example_row = only_zero.iloc[4]

    v = example_row["verb"]
    v_c = example_row["verb_compound"]
    m_c = example_row["morph_case"]

    # transaction_head.form as head_form, lemma, spatial_obl.form as verb_form, verb, verb_compound, morph_case, sentence_id, sentence, phrase
    query = f"""SELECT head_id, head_form, head_lemma, tbl2.form as verb_form, tbl1.verb, tbl1.verb_compound, 
                tbl1.morph_case, tbl1.sentence_id, tbl1.sentence, tbl2.phrase, tbl1.ekilex_tag

                FROM (
                SELECT head_id, form as head_form, lemma as head_lemma, verb, verb_compound, morph_case, 
                sentence_id, sentence, ekilex_tag 
                FROM spatial_obl 
                where 
                verb = '{v}' and 
                verb_compound='{v_c}' and 
                morph_case='{m_c}'
                ) as tbl1

                join

                (SELECT * from transaction_head) as tbl2 on 
                tbl1.verb = tbl2.verb and 
                tbl1.verb_compound = tbl2.verb_compound and 
                tbl1.sentence_id = tbl2.sentence_id
                """

    spatial_obl_ex = pd.read_sql(query, conn)
    df2 = spatial_obl_ex[spatial_obl_ex["ekilex_tag"]=='location'] # saab kõik location näited
    df3 = df2.head(10) # 10 näidet kui on location
    # 10 näidet kui ei ole location
    df2_2 = spatial_obl_ex[(spatial_obl_ex["ekilex_tag"]!='location') & (spatial_obl_ex["ekilex_tag"].notna())] # 
    df3_2 = df2_2.head(10)
    
    df4 = pd.concat([df3, df3_2], ignore_index=True)
    
    
    big_df = pd.concat([big_df, df4], ignore_index=True)

In [63]:
big_df # kui võtta kõik siis 105070 näidet

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag
0,34594,kodus,kodu,valitseb,valitsema,,in,20101,"Õnneks piisab mulle sellest , kui mina ise ja ...",kodus valitseb armastus,location
1,39275,haiglas,haigla,valitses,valitsema,,in,22819,Mustamäe haiglas valitses Bentonite hinnangul ...,haiglas valitses hinnangul teadmatus,location
2,39620,Tööhõiveametis,tööhõiveamet,valitseb,valitsema,,in,23033,“ Tööhõiveametis valitseb täielik tegematajätm...,Tööhõiveametis valitseb tegematajätmine,location
3,72824,Muusikakoolis,muusikakool,valitses,valitsema,,in,42060,“ Muusikakoolis valitses reaalõpetajate terror...,Muusikakoolis valitses terror,location
4,83937,tööhõiveametis,tööhõiveamet,valitseb,valitsema,,in,48642,Raidi arvates valitseb tööhõiveametis töö korr...,arvates valitseb tööhõiveametis korraldamatus,location
...,...,...,...,...,...,...,...,...,...,...,...
575,1408398,novembris,november,puhkes,puhkema,,in,884383,Eesti üheks eliitkooliks peetavas Hugo Treffne...,gümnaasiumis puhkes tulekahju novembris,time
576,1436524,novembris,november,puhkes,puhkema,,in,901833,1808. aastal lõppes Rootsi ülemvõim ja sama aa...,novembris puhkes Helsingis tulekahju,time
577,1493775,detsembris,detsember,puhkenud,puhkema,,in,938116,Valge maja ja teiste riigiasutuste hoonete ümb...,detsembris Seattle'is puhkenud,time
578,1528461,septembris,september,puhkes,puhkema,,in,959426,""" Eelmise aasta septembris puhkes Kastre metsk...",septembris puhkes metskonnas vallas põleng,time


In [75]:
big_df.to_csv("n80_top29_10p_10n_examples.csv", encoding="utf-8", index = False, sep="|")

In [ ]:
conn.close()

# Kui on eelnevalt salvestatud csv siis lugeda sisse

In [2]:
spatial_obl_ex = pd.read_csv("n80_top29_10p_10n_examples.csv", encoding="utf-8",  sep="|")

In [3]:
def eki_loc(row):
    if row['ekilex_tag'] == 'location':
        return "yes"
    else:
        return "no"

In [4]:
spatial_obl_ex["eki_is_loc"] = spatial_obl_ex.apply(eki_loc, axis=1)

In [5]:
spatial_obl_ex

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag,eki_is_loc
0,34594,kodus,kodu,valitseb,valitsema,NaN,in,20101,"Õnneks piisab mulle sellest , kui mina ise ja ...",kodus valitseb armastus,location,yes
1,39275,haiglas,haigla,valitses,valitsema,NaN,in,22819,Mustamäe haiglas valitses Bentonite hinnangul ...,haiglas valitses hinnangul teadmatus,location,yes
2,39620,Tööhõiveametis,tööhõiveamet,valitseb,valitsema,NaN,in,23033,“ Tööhõiveametis valitseb täielik tegematajätm...,Tööhõiveametis valitseb tegematajätmine,location,yes
3,72824,Muusikakoolis,muusikakool,valitses,valitsema,NaN,in,42060,“ Muusikakoolis valitses reaalõpetajate terror...,Muusikakoolis valitses terror,location,yes
4,83937,tööhõiveametis,tööhõiveamet,valitseb,valitsema,NaN,in,48642,Raidi arvates valitseb tööhõiveametis töö korr...,arvates valitseb tööhõiveametis korraldamatus,location,yes
...,...,...,...,...,...,...,...,...,...,...,...,...
575,1408398,novembris,november,puhkes,puhkema,NaN,in,884383,Eesti üheks eliitkooliks peetavas Hugo Treffne...,gümnaasiumis puhkes tulekahju novembris,time,no
576,1436524,novembris,november,puhkes,puhkema,NaN,in,901833,1808. aastal lõppes Rootsi ülemvõim ja sama aa...,novembris puhkes Helsingis tulekahju,time,no
577,1493775,detsembris,detsember,puhkenud,puhkema,NaN,in,938116,Valge maja ja teiste riigiasutuste hoonete ümb...,detsembris Seattle'is puhkenud,time,no
578,1528461,septembris,september,puhkes,puhkema,NaN,in,959426,""" Eelmise aasta septembris puhkes Kastre metsk...",septembris puhkes metskonnas vallas põleng,time,no


# OSA II : GPT

## GPT jaoks vajalik

In [6]:
config = configparser.ConfigParser()
status = config.read('azure.ini') 
assert status == ['azure.ini']

API_VERSION = config['azure-configuration']['api_version']
AZURE_ENDPOINT = config['azure-configuration']['api_base']
SUBSCRIPTION_KEY = config['azure-configuration']['api_key']
model_name = "gpt-4o" #"GPT-4o-2024-1120 Global"
DEPLOYMENT = config['azure-configuration']['deployment_id']

In [7]:
def in2json(sisend):
    return json.dumps(sisend, ensure_ascii=False)

In [8]:
SYSTEM_PROMPT = """
You are a classification assistant.
Your task: Given a list of JSON objects, each with keys "l" (sentence) and "c" (phrase), classify whether "c" is a location in the context of the sentence.
Locations: names of locations, buildings and bussinesses (bankhouse, studio, club), physical objects and living things, areas that have a defined geographic location.
Not locations:abstract places, actions and events, living beings who are action takers, state of being, ordinance, causal regulation, adverb of time, constructions and stamp expressions.
Output requirements:
- Respond with an array of JSON objects, one per input item.
- The output array must be in the exact same order as the input items.
- Each output object must have:
  "a": "yes" (location) or "no" (not location),
  "s": a one-word category such as "city", "country", "location", "time", "org", "object", "event", "abstract", "person", "activity",
  "r": one sentence explaining why phrase is location or not.
Rules:
- Strict JSON only.
- No commentary.
- No markdown.
- No merging of items — one output per input.
"""

FEW_SHOTS = [
            {
            "role": "user",
            "content": in2json({"l": "Me läksime Pariisi", "c": "Pariisi"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "city", "r": "Phrase is a city name and therefore location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Ema on mul olnud alati õmblustöö inimene ja õpetab seda praegu ühes õmbluskoolis teistelegi.", "c": "õmbluskoolis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "building", "r": "Phrase is a buidlding but also a bussiness."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Põhjapoolusele saabus kottide-kompsudega tuhandeid võõrtöölisi.", "c": "Põhjapoolusele"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "location", "r": "Phrase is an area with defined geographical location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Ta tuli idast kõikide oma raamatutega.", "c": "idast"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "direction", "r": "Phrase is an area with defined geographical location in the context of this sentence."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Mees istus peale pikka päeva uuesti sadulasse.", "c": "sadulasse"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "yes", "s": "object", "r": "Phrase is an object that can be defines as location."})
            },
    
            {
            "role": "user",
            "content": in2json({"l": "Ta alustas tööd kell üheksa", "c": "kell üheksa"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "time", "r": "Phrase is time expression not location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Avo Mäeseppa süüdistati selles, et ta eelmise aasta sügisel varastas magava J.P. põuetaskust salaja raha koos rahakotiga.", "c": "põuetaskust"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "abstract", "r": "The theft did not happen in põuetasku and therefore is not a location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo restoranis Park.", "c": "restoranis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "abstract", "r": "The party is planned to be in a restaurant but the action of planning is not happening there."})
            },

            {
            "role": "user",
            "content": in2json({"l": "HP700 ei ulatu enam SpeedTouchi Wifi'sse.", "c": "Wifi'sse"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "abstract", "r": "The geographic location of the phrase acn't be determined."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Minnie käis Barbra teadmata isegi kleidiproovis.", "c": "kleidiproovis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "event", "r": "Phrase is an event."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Rüselejal käsisid sussid", "c": "Rüselejal"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "actor", "r": "Phrase refers to action taker."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Nüüd siis istun sitas.", "c": "sitas"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "state", "r": "Phrase refers to state of being."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Protest on mitmekesine ja teravaimalt avaldub see kirjanduses.", "c": "teravaimalt"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "ordinance", "r": "Phrase refers to ordinance and is not location."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Tulbisibul on siinkohal platseebo ja selle hävitamisel kaovad ka sümptomid.", "c": "hävitamisel"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "regulation", "r": "The phrase is causal regulation."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Rahulepingu kehtivusest lähtus omariikluse taastamise käigus Ülemnõukogu.", "c": "kehtivusest"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "construction", "r": "The phrase refers to construction of stamp expression."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Korraldasime seminari TTÜs.", "c": "TTÜs"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "org", "r": "The phrase refers to organization."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Väga hästi varjab päikesekiiri näiteks markiis.", "c": "markiis"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "object", "r": "The phrase refers to an object that is in the inessive case but is not the location of the action."})
            },

            {
            "role": "user",
            "content": in2json({"l": "Ingridi puhul läks hiljem täkkesse just see ütelus.", "c": "täkkesse"})
            },
            {
            "role": "assistant",
            "content": in2json({"a": "no", "s": "stamp", "r": "The phrase is in illativa case but is a stamp expression."})
            },

]
 

## tokenite arvutuseks

In [ ]:
# example batch

In [172]:
batch_items = []

for i in range(len(spatial_obl_ex)):
    ex = spatial_obl_ex.iloc[i]
    batch_items.append(in2json({"l": ex["sentence"], "c": ex["head_form"]}))

    if i==10:
        break

In [173]:
batch_items

['{"l": "Õnneks piisab mulle sellest , kui mina ise ja minu lähedased on terved ning kodus valitseb armastus .", "c": "kodus"}',
 '{"l": "Mustamäe haiglas valitses Bentonite hinnangul tume teadmatus .", "c": "haiglas"}',
 '{"l": "“ Tööhõiveametis valitseb täielik tegematajätmine , ” pragab Raid .", "c": "Tööhõiveametis"}',
 '{"l": "“ Muusikakoolis valitses reaalõpetajate terror , nad ei suutnud muusikute hinge mõista .", "c": "Muusikakoolis"}',
 '{"l": "Raidi arvates valitseb tööhõiveametis töö korraldamatus .", "c": "tööhõiveametis"}',
 '{"l": "Täpselt niisugune mentaliteet valitses Ukrainas .", "c": "Ukrainas"}',
 '{"l": "Leidmaks lahendust Jaan Tõnissoni kadumisele , tuleks tagasi pöörduda olukorra juurde , mis valitses Tallinnas 1941. aasta juuli alguspäevadel .", "c": "Tallinnas"}',
 '{"l": "Pärast seda kui iseseisvus käes , valitses Eestis mõnd aega aateline rahvuslus , laulva revolutsiooni järgne palang .", "c": "Eestis"}',
 '{"l": "“ Eestis valitseb väga primitiivne arusaam ava

In [174]:
user_payload = {
        "Instruction": (
            "First interpret the few-shot examples. "
            "Then process the list called 'batch'. "
            "Output a JSON array with one item per batch entry, in the same order."
        ),
        "few_shots": FEW_SHOTS,
        "batch": batch_items
    }

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": in2json(user_payload)}
]

# oletame et mudeli output on sama palju tokeneid kui batch items


### manuaalne umbkaudne sisend

In [175]:
try:
    enc = tiktoken.encoding_for_model("gpt-4o")
except KeyError:
    enc = tiktoken.get_encoding("o200k_base")

def count_tokens(text):
    return len(enc.encode(text))

def count_message_tokens(messages):
    total = 0
    for m in messages:
        total += len(enc.encode(m["role"]))
        total += len(enc.encode(m["content"]))
    return total

In [176]:

print("KOKKU tokeneid:", count_message_tokens(messages))
print("System prompt tokeneid:", len(enc.encode(SYSTEM_PROMPT)))
print("Few-shots tokeneid:", count_message_tokens(FEW_SHOTS))
print("väljund tokeneid:", len(enc.encode(" ". join(batch_items))))
print(count_message_tokens(messages)+len(enc.encode(" ". join(batch_items))))

KOKKU tokeneid: 2222
System prompt tokeneid: 256
Few-shots tokeneid: 1078
väljund tokeneid: 471
2693


In [72]:
#400*580 # muidu oleks 105070 kirjet 580 asemel

232000

In [177]:
2693*58

156194

### cost estimator

In [178]:
messages2 = messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": in2json(user_payload)},
    {"role": "assistant", "content":in2json(batch_items)}
]

In [179]:
@CostEstimator()
def query_openai(model, messages, **kwargs):
    args_to_remove = ['mock', 'completion_tokens']

    for arg in args_to_remove:
        if arg in kwargs:
            del kwargs[arg]

    return openai.ChatCompletion.create(
        model = model,
        messages = messages,
        **kwargs)


responses = []
i = 0
#for i in tqdm(range(0,1)):
response = query_openai(
  model="gpt-4o",
  messages = messages,
  temperature=0,
  mock=True,
  completion_tokens=1
)

responses.append({
      'input': i,
      'output': response["choices"][0]["message"]["content"]
    })

print() # Empty line to display the total sum

# Print the responses
#print(responses)

Cost: $0.0075 | Total: $0.0426


In [81]:
0.0426*58 # -> eurodes

2.4708

In [181]:
CostEstimator.get_total_cost(CostEstimator)

0.0425875

In [182]:
CostEstimator.reset()

## pydantic

In [9]:

class ClassificationDict(BaseModel):
    a: Literal["yes", "no"]
    s: str
    r: str


In [10]:
client = AzureOpenAI(
    api_version=API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
    api_key=SUBSCRIPTION_KEY,
)

## Andmete söötmine

In [21]:
def classify(my_batch):

    print("classify", len(my_batch))
    max_att = 2
    attempt = 0
    while attempt < max_att:
        attempt += 1
        user_payload = {
            "Instruction": (
                "Analyse the few-shot examples. "
                "Then process the list called 'batch'. "
                "Output a JSON array with one item per batch entry, in the same order."
                "Output a JSON array of EXACTLY N items (same length as 'batch' list) in the same order. Do not add or remove items."
            ),
            "few_shots": FEW_SHOTS,
            "batch": in2json(my_batch)
        }
    
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": in2json(user_payload)}
        ]

        response = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=messages,
            temperature=0 # absoluutselt min väljund ehk tahan 1 tokenit
        )

        raw_output = response.choices[0].message.content.strip()

        try:
            data = json.loads(raw_output)

            if len(data) != len(batch):
                raise ValueError("Väljundis ei ole õige arv vastuseid.")
                
            elif len(data) == len(batch):
                for item in data:
                    ClassificationDict(**item)

            return response, raw_output

        except (ValidationError, json.JSONDecodeError, ValueError) as e:
            print(f"Attempt {attempt} failed. Retrying batch...")
            print(f"Error: {e}")
            #print(f"Raw output: {raw_output[:500]}...")  # preview first 500 chars
            time.sleep(1)  # small delay before retry

    print(f"Batch failed after {max_att} attempts.")

    return response, raw_output

In [34]:
df = spatial_obl_ex.sample(frac=1)#.reset_index(drop=True)


In [35]:
results = []
responses = []
batch = []

used_tokens = 0

# kui tahta kõiki näiteid anda gpt-le
for j in range(len(df)):

    if len(batch) < 10:
        ex = df.iloc[j]
        batch.append(in2json({"l": ex["sentence"], "c": ex["head_form"]}))
        
    if len(batch) == 10 or (len(batch) < 10 and j==len(df)-1):
        response, result = classify(batch)
        results.append(result)
        responses.append(response)
        used_tokens += response.usage.total_tokens
        batch = []
        #break



classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10
classify 10


In [36]:
used_tokens # ümardame 580 lauset batchiga 10 on u 160K tokenit responsi põhjal

158774

In [38]:
answers = []
shorts = []
long = []

problematic = []

bs = 10

for elem in results:
    try:
        #print(len(elem.split("\n")))
        data = json.loads(elem)
        if len(data) != bs:
            for i in range(bs):
                answers.append("?")
                shorts.append("?")
                long.append("?")
            problematic.append(data)
        else:
            for item in data:
                answers.append(item["a"])
                shorts.append(item["s"])
                long.append(item["r"])
    except Exception as e:
        #print(len(elem.split("\n")))
        print(elem)
        #continue

In [42]:
df["gpt_is_loc"] = answers
df["short_answ"] = shorts
df["long_answ"] = long

In [46]:
df

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag,eki_is_loc,gpt_is_loc,short_answ,long_answ
528,89588,maakodus,maakodu,peatub,peatuma,NaN,in,51781,"Aivar peatub Helve maakodus Sauel , kus nende päevad mööduvad looduses jalutades ja aiatööd tehes .",Aivar peatub maakodus,location,yes,yes,location,Phrase refers to a specific location where the action takes place.
13,304611,suhtumises,suhtumine,valitseb,valitsema,NaN,in,183264,Terav kontrast valitseb ka suhtumises lähimasse läänenaabrisse .,kontrast valitseb suhtumises,not_location,no,no,abstract,Phrase refers to an abstract concept and not a physical location.
487,147576,Bagdadist,Bagdad,naasnud,naasma,NaN,el,86827,"Kümneid kordi rohkem kuluks aga siis , kui ÜRO peasekretär Kofi Annan poleks Bagdadist naasnud kokkuleppega , mille tagajärgede , sisu ja otstarbekuse üle vaieldakse siiamaani .",peasekretär poleks Bagdadist naasnud kokkuleppega,location,yes,yes,city,"Phrase refers to a city, which is a defined geographic location."
355,11986,trennis,trenn,käib,käima,NaN,in,6930,Ta käib nimelt paar korda nädalas Tartu Arenas nii beebiga koos kui ka naiste trennis ning ega intensiivne elulaad lase üheski mõttes lõtvuda .,Ta käib nimelt korda Arenas koos trennis,event,no,no,activity,Phrase refers to an activity and not a location.
243,17810,Mehhikos,Mehhiko,juhtus,juhtuma,NaN,in,10414,See juhtus Mehhikos rannas .,See juhtus Mehhikos rannas,location,yes,yes,country,"Phrase refers to a country, which is a defined geographic location."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
194,410832,augustis,august,korraldasime,korraldama,NaN,in,260305,"2002. aasta augustis korraldasime TPÜ-s ühisseminari koos FLE3 arendusmeeskonnaga ja saime põhjaliku ülevaate FLE3 aluseks olevatest pedagoogilistest ideedest , süsteemi ülesehitusest , funktsionaalsusest ja eripäradest .",augustis korraldasime TPÜ-s ühisseminari koos arendusmeeskonnaga,time,no,no,time,"Phrase refers to a time expression, not a location."
322,85130,Eestis,Eesti,puuduvad,puuduma,NaN,in,49284,"Kahjuks ei pruugi areng toimuda lähima kümne-kahekümne aasta jooksul , sest Eestis puuduvad vajalikud miljardid .",Eestis puuduvad miljardid,location,yes,yes,country,Phrase refers to a country and is therefore a location.
561,107618,Tartus,Tartu,puhkenud,puhkema,NaN,in,62630,"Nii kirjutanud lehed liiga palju 18. juunil 1937 Tartus Kauba tänaval puhkenud tulekahjust , mistõttu Tartu linnapea tundnud end puudutatuna ja teinud sellest siseministrile ettekande , leides , et Tartu linnavalitsust on asjata süüdistatud .",Tartus tänaval puhkenud,location,yes,yes,city,Phrase refers to a city and is therefore a location.
122,111267,Poolas,Poola,avanud,avama,NaN,in,64771,Daewoo on Poolas tütarettevõtte avanud ning poolakad saavad ematehases väljaõpet .,Daewoo on Poolas tütarettevõtte avanud,location,yes,yes,country,Phrase refers to a country and is therefore a location.


In [44]:
df.to_csv("n80_top29_10p_10n_examples_gpt_v2_uus_prompt_batch10.csv", encoding="utf-8", index = False, sep="|")

In [47]:
df = pd.read_csv("n80_top29_10p_10n_examples_gpt_v2_uus_prompt_batch10.csv", encoding="utf-8",  sep="|")

In [49]:
no_match = df[df["eki_is_loc"]!=df["gpt_is_loc"]]

In [50]:
len(no_match)

36

In [57]:
len(no_match[no_match["head_lemma"]!="pai"])

27

In [63]:
len(df[(df["ekilex_tag"]=='location') & (df["gpt_is_loc"]=='yes')]) # 279/290

279

In [64]:
len(df[(df["ekilex_tag"]!='location') & (df["gpt_is_loc"]=='no')]) # 265/290

265

In [59]:
no_pai = no_match[no_match["head_lemma"]!="pai"]

In [61]:
no_match[(no_match["ekilex_tag"]=='location') & (no_match["gpt_is_loc"]=='no')] # 11 juhtu

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag,eki_is_loc,gpt_is_loc,short_answ,long_answ
9,65899,koju,kodu,tõime,tooma,NaN,adit,38057,"Kui elasime Tartu eeslinnas Haagel , tõime tihtilugu lapsed koju ja sõitsime Tartusse tagasi .",elasime tõime tihtilugu lapsed koju,location,yes,no,abstract,Phrase refers to an abstract concept and not a physical location.
76,658899,naabrusest,naabrus,varastasid,varastama,NaN,el,422654,"Seni teadmata kurjategijad varastasid eile öösel Russalka kuju naabrusest seitse mustast malmist otstega diivanpinki , nädalapäevad varem jäi kolmest säärasest pingist ilma ka Metsakalmistu bussipeatus .",kurjategijad varastasid eile öösel kuju naabrusest diivanpinki,location,yes,no,location,Phrase refers to an area near a location but is not a specific location itself.
90,72267,restoranis,restoran,korraldas,korraldama,NaN,in,41742,Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo restoranis Park .,Näiteks korraldas Affleck restoranis Park,location,yes,no,abstract,The party is planned to be in a restaurant but the action of planning is not happening there.
98,83937,tööhõiveametis,tööhõiveamet,valitseb,valitsema,NaN,in,48642,Raidi arvates valitseb tööhõiveametis töö korraldamatus .,arvates valitseb tööhõiveametis korraldamatus,location,yes,no,org,"Phrase refers to an organization, not a location."
128,43450,siitilmast,siitilm,lahkus,lahkuma,NaN,el,25136,"Kui tal ka oli õnnestunud poja mõrvamise asjaoludele lähemale jõuda , lahkus ta siitilmast ilmselt koos oma saladustega .",õnnestunud lahkus ta siitilmast ilmselt koos saladustega,location,yes,no,abstract,"Phrase refers to an abstract concept of leaving the world, not a physical location."
208,17168,süles,süli,istub,istuma,NaN,in,10010,Ühel fotol istub ooperitäht Pirjo Levandi kaksiratsi oma muusikust elukaaslase Jüri Peetsoni süles ning nad suudlevad .,fotol istub ooperitäht kaksiratsi muusikust süles,location,yes,no,object,The phrase refers to an object and not a location.
305,16143,koju,kodu,tõi,tooma,NaN,adit,9424,"Sama kordus , kui Mark tõi esimese kursuse kevadel Moskvast koju mongoollanna ja tutvustas teda oma noorikuna .",Mark tõi kursuse kevadel Moskvast koju mongoollanna,location,yes,no,abstract,"Phrase refers to an abstract concept of returning home, not a specific location."
337,8175,tippklubisse,tippklubi,Jõudis,jõudma,NaN,ill,4723,"Jõudis tippklubisse ning hakkas siis , juba 24-aastaselt , täiskohaga treeneriks .",Jõudis tippklubisse,location,yes,no,org,"Phrase refers to an organization, not a location."
365,177344,Nõukogus,nõukogu,tegelevad,tegelema,NaN,in,105165,"Ühe prioriteedi eelisarendamine poleks ka võimalik , sest Nõukogus tegelevad erinevate valdkondadega komisjonid , mis oma tööd ei katkesta .",Nõukogus tegelevad komisjonid,location,yes,no,org,"Phrase refers to an organization, not a location."
452,39620,Tööhõiveametis,tööhõiveamet,valitseb,valitsema,NaN,in,23033,"“ Tööhõiveametis valitseb täielik tegematajätmine , ” pragab Raid .",Tööhõiveametis valitseb tegematajätmine,location,yes,no,org,"Phrase refers to an organization, not a physical location."


In [70]:
no_match[(no_match["ekilex_tag"]!='location') & (no_match["gpt_is_loc"]=='yes')] # 25, nendest 9 on Paide juhud

,head_id,head_form,head_lemma,verb_form,verb,verb_compound,morph_case,sentence_id,sentence,phrase,ekilex_tag,eki_is_loc,gpt_is_loc,short_answ,long_answ
66,4458263,spordilaagrisse,spordilaager,olnud,olema,NaN,ill,2780022,"Jordaaniasse spordilaagrisse teel olnud 15 idamaiste võitluskunstide meistrit rööviti eelmise aasta mais Bagdadi lähistel Anbari provintsis , vahendab BBC .",Jordaaniasse spordilaagrisse teel olnud,event,no,yes,location,The phrase refers to a specific place where the sports camp is located.
73,386259,presiidiumis,presiidium,Istusin,istuma,NaN,in,241521,"Istusin koos paari rahvasaadiku ja NLKP ideoloogiaosakonna juhatajaga presiidiumis , meile esitati küsimusi .",Istusin koos rahvasaadiku presiidiumis,not_location,no,yes,location,"Phrase refers to a specific place, a presiding area, which is a location."
83,4891967,Paides,pai,süttis,süttima,NaN,in,3049202,"PAIDE , 29. detsember ( EPLO ) - Teisipäeva keskpäeval süttis Paides ilmselt laste ilutulestikuga mängimisest kolmekorruselise elamu korter .",PAIDE keskpäeval süttis Paides ilmselt mängimisest korter,not_location,no,yes,city,Phrase is a city name and therefore a location.
86,4945,trennis,trenn,käib,käima,NaN,in,2965,"Meil käib trennis ka Kuuba ja Maroco sportlasi , neisse suhtutakse veel halvemini .",Meil käib trennis sportlasi,event,no,yes,location,"Phrase refers to a place where training occurs, which is a location."
101,208011,Paides,pai,sündinud,sündima,NaN,in,124135,Viimasel kolmel aastal on Paides sündinud vaid 100 last aastas .,aastal on Paides sündinud last,not_location,no,yes,city,Phrase is a city name and therefore a location.
113,3471266,ründelauas,ründelaud,möllas,möllama,NaN,in,2172644,Tein murdis treener Lauri rõõmuks raginal sisse ja Reinkort möllas ründelauas .,Reinkort möllas ründelauas,not_location,no,yes,object,Phrase refers to a physical object that can be considered a location in the context of the sentence.
165,1660687,pioneerilaagrites,pioneerilaager,kasvanud,kasvama,üles,in,1044272,"Rannasaare sõnul toob osa vanemaid lapsed laagrisse ka nostalgiast , sest on ise suvistes pioneerilaagrites üles kasvanud .",on ise pioneerilaagrites üles kasvanud,event,no,yes,location,Phrase refers to a specific place where the action occurs.
237,220518,Paidest,pai,saabus,saabuma,NaN,el,131691,"Esimestena olidki kohal päästeameti töötajad , 10-15 minuti jooksul saabus Paidest kiirabi , hoopis hiljem jõudis kohale reanimobiil Tallinnast .",minuti jooksul saabus Paidest kiirabi,not_location,no,yes,city,"Phrase refers to a city, which is a location."
239,2163524,Koreas,korea,peetud,peetuma,NaN,in,1361022,"Ranskalaiste Tartus elavad sõbrad on sageli maininud , et järjekordne pidu või sünnipäev sai peetud Koreas .",pidu sai peetud Koreas,state,no,yes,country,"Phrase refers to a country, which is a location."
246,172633,Paidesse,pai,viib,viima,NaN,ill,102087,"Tema tee viib kõigepealt Tallinnast Paidesse ja sealt Ameerikasse , kus Carmen suure osa oma ajast veedab .",tee viib kõigepealt Tallinnast Paidesse,not_location,no,yes,city,"Phrase refers to Paidesse, which is a city and a location."


In [77]:
tbl = no_match[["head_form", "head_lemma", "verb", "morph_case", "sentence", "phrase", "ekilex_tag", "gpt_is_loc", "short_answ", "long_answ"]]

In [78]:
tbl[(tbl["ekilex_tag"]!='location') & (tbl["gpt_is_loc"]=='yes') & (tbl["head_lemma"]!="pai")] # 16 juhtu, 4 juhtu ekilex on vale

,head_form,head_lemma,verb,morph_case,sentence,phrase,ekilex_tag,gpt_is_loc,short_answ,long_answ
66,spordilaagrisse,spordilaager,olema,ill,"Jordaaniasse spordilaagrisse teel olnud 15 idamaiste võitluskunstide meistrit rööviti eelmise aasta mais Bagdadi lähistel Anbari provintsis , vahendab BBC .",Jordaaniasse spordilaagrisse teel olnud,event,yes,location,The phrase refers to a specific place where the sports camp is located.
73,presiidiumis,presiidium,istuma,in,"Istusin koos paari rahvasaadiku ja NLKP ideoloogiaosakonna juhatajaga presiidiumis , meile esitati küsimusi .",Istusin koos rahvasaadiku presiidiumis,not_location,yes,location,"Phrase refers to a specific place, a presiding area, which is a location."
86,trennis,trenn,käima,in,"Meil käib trennis ka Kuuba ja Maroco sportlasi , neisse suhtutakse veel halvemini .",Meil käib trennis sportlasi,event,yes,location,"Phrase refers to a place where training occurs, which is a location."
113,ründelauas,ründelaud,möllama,in,Tein murdis treener Lauri rõõmuks raginal sisse ja Reinkort möllas ründelauas .,Reinkort möllas ründelauas,not_location,yes,object,Phrase refers to a physical object that can be considered a location in the context of the sentence.
165,pioneerilaagrites,pioneerilaager,kasvama,in,"Rannasaare sõnul toob osa vanemaid lapsed laagrisse ka nostalgiast , sest on ise suvistes pioneerilaagrites üles kasvanud .",on ise pioneerilaagrites üles kasvanud,event,yes,location,Phrase refers to a specific place where the action occurs.
239,Koreas,korea,peetuma,in,"Ranskalaiste Tartus elavad sõbrad on sageli maininud , et järjekordne pidu või sünnipäev sai peetud Koreas .",pidu sai peetud Koreas,state,yes,country,"Phrase refers to a country, which is a location."
284,tantsutrennis,tantsutrenn,käima,in,"Jah , kui mu sõbrannadel olid esimesed käest kinni hoidmise suhted 13aastaselt , käisin mina tantsutrennis .",Jah olid käisin mina tantsutrennis,event,yes,activity,Phrase refers to a specific activity that takes place in a defined location.
334,Permis,perm,treenima,in,"Kui enne mänge Permis treenisid klubide koondislased koos vaid korra , siis tänase Tallinna-tuuri eel on liikumisi lihvitud viis korda .",enne mänge Permis treenisid koondislased koos korra,time,yes,city,"Phrase refers to a city, which is a defined geographic location."
345,trennis,trenn,käima,in,Praegu käin trennis kolm korda nädalas .,Praegu käin trennis korda,event,yes,activity,Phrase refers to a place where an activity occurs and is therefore a location.
351,Koreas,korea,tegelema,in,"Lisaks kunstile tegeleb Altnurme Koreas ka näitlemisega , mängides kohalikus seebiseriaalis .",Lisaks tegeleb Koreas näitlemisega mängides,state,yes,country,"Phrase refers to a country, which is a location."


In [76]:
tbl[(tbl["ekilex_tag"]=='location') & (tbl["gpt_is_loc"]=='no') & (tbl["head_lemma"]!="pai")] # 11

,head_form,head_lemma,verb,sentence,phrase,ekilex_tag,gpt_is_loc,short_answ,long_answ
9,koju,kodu,tooma,"Kui elasime Tartu eeslinnas Haagel , tõime tihtilugu lapsed koju ja sõitsime Tartusse tagasi .",elasime tõime tihtilugu lapsed koju,location,no,abstract,Phrase refers to an abstract concept and not a physical location.
76,naabrusest,naabrus,varastama,"Seni teadmata kurjategijad varastasid eile öösel Russalka kuju naabrusest seitse mustast malmist otstega diivanpinki , nädalapäevad varem jäi kolmest säärasest pingist ilma ka Metsakalmistu bussipeatus .",kurjategijad varastasid eile öösel kuju naabrusest diivanpinki,location,no,location,Phrase refers to an area near a location but is not a specific location itself.
90,restoranis,restoran,korraldama,Näiteks korraldas Affleck Lopezile üllatus-sünnipäevapeo restoranis Park .,Näiteks korraldas Affleck restoranis Park,location,no,abstract,The party is planned to be in a restaurant but the action of planning is not happening there.
98,tööhõiveametis,tööhõiveamet,valitsema,Raidi arvates valitseb tööhõiveametis töö korraldamatus .,arvates valitseb tööhõiveametis korraldamatus,location,no,org,"Phrase refers to an organization, not a location."
128,siitilmast,siitilm,lahkuma,"Kui tal ka oli õnnestunud poja mõrvamise asjaoludele lähemale jõuda , lahkus ta siitilmast ilmselt koos oma saladustega .",õnnestunud lahkus ta siitilmast ilmselt koos saladustega,location,no,abstract,"Phrase refers to an abstract concept of leaving the world, not a physical location."
208,süles,süli,istuma,Ühel fotol istub ooperitäht Pirjo Levandi kaksiratsi oma muusikust elukaaslase Jüri Peetsoni süles ning nad suudlevad .,fotol istub ooperitäht kaksiratsi muusikust süles,location,no,object,The phrase refers to an object and not a location.
305,koju,kodu,tooma,"Sama kordus , kui Mark tõi esimese kursuse kevadel Moskvast koju mongoollanna ja tutvustas teda oma noorikuna .",Mark tõi kursuse kevadel Moskvast koju mongoollanna,location,no,abstract,"Phrase refers to an abstract concept of returning home, not a specific location."
337,tippklubisse,tippklubi,jõudma,"Jõudis tippklubisse ning hakkas siis , juba 24-aastaselt , täiskohaga treeneriks .",Jõudis tippklubisse,location,no,org,"Phrase refers to an organization, not a location."
365,Nõukogus,nõukogu,tegelema,"Ühe prioriteedi eelisarendamine poleks ka võimalik , sest Nõukogus tegelevad erinevate valdkondadega komisjonid , mis oma tööd ei katkesta .",Nõukogus tegelevad komisjonid,location,no,org,"Phrase refers to an organization, not a location."
452,Tööhõiveametis,tööhõiveamet,valitsema,"“ Tööhõiveametis valitseb täielik tegematajätmine , ” pragab Raid .",Tööhõiveametis valitseb tegematajätmine,location,no,org,"Phrase refers to an organization, not a physical location."


In [80]:
tbl[(tbl["ekilex_tag"]!='location') & (tbl["gpt_is_loc"]=='no')] 

,head_form,head_lemma,verb,morph_case,sentence,phrase,ekilex_tag,gpt_is_loc,short_answ,long_answ
